Instalación para Transformers

In [ ]:
# Esto es pesado (~2GB la primera vez que descarga el modelo)
pip install transformers torch datasets accelerate

# Si tienes GPU NVIDIA:
pip uninstall torch -y
pip install torch --index-url https://download.pytorch.org/whl/cu124

# Para verificar si tienes GPU disponible:
python -c "import torch; print(f'CUDA disponible: {torch.cuda.is_available()}')"

 Cargar datos para el Transformer

 Los Transformers NO usan TF-IDF. Tienen su propio tokenizador
que convierte texto en IDs numéricos que el modelo entiende.

"profit fell sharply" → [2082, 3062, 13559] (IDs del vocabulario)

La diferencia clave: el tokenizador mantiene el ORDEN y la
POSICIÓN de cada palabra. TF-IDF perdía eso.

In [5]:

import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# 1. Cargar datos (mismo CSV de Kaggle)
df = pd.read_csv("dataset/all-data.csv",
                 encoding="latin-1",
                 header=None,
                 names=["sentiment", "sentence"])

# 2. Convertir etiquetas a números
#    Los Transformers necesitan labels numéricos
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["sentiment"].map(label2id)

# 3. Train / Validation / Test split
#    Ahora hacemos 3 partes: train para entrenar, val para ajustar
#    hiperparámetros DURANTE el entrenamiento, test para evaluar al final.
#    ¿Por qué val? Porque el Transformer entrena varias "épocas" y
#    necesitamos saber cuándo parar (si no, sobreajusta).
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print(f"Train: {len(train_df)} frases")
print(f"Val:   {len(val_df)} frases")
print(f"Test:  {len(test_df)} frases")

# 4. Convertir a formato HuggingFace Dataset
#    Es como un DataFrame pero optimizado para entrenar modelos
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["sentence", "label"]].reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df[["sentence", "label"]].reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df[["sentence", "label"]].reset_index(drop=True)),
})

print(f"\nDataset creado:")
print(dataset)

# Veamos un ejemplo
print(f"\nEjemplo del dataset:")
print(dataset["train"][0])

Train: 3392 frases
Val:   727 frases
Test:  727 frases

Dataset creado:
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3392
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 727
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 727
    })
})

Ejemplo del dataset:
{'sentence': 'In the Baltic countries , sales fell by 42.6 % .', 'label': 0}


Tokenización
Este bloque es donde el Transformer empieza a diferenciarse. Su tokenizador no solo parte en palabras — usa subword tokenization: puede partir una palabra desconocida en trozos que sí conoce.

In [6]:
from transformers import AutoTokenizer

# Descargar tokenizador de FinBERT (~500MB primera vez)
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Veamos cómo tokeniza
examples = [
    "Profit increased significantly in the third quarter",
    "Operating losses widened due to restructuring charges",
    "The annual general meeting will be held next month",
]

print("CÓMO TOKENIZA FINBERT:")
print("=" * 60)
for text in examples:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text)
    print(f"\n  Texto:   {text}")
    print(f"  Tokens:  {tokens}")
    print(f"  IDs:     {ids}")
    print(f"  N tokens: {len(tokens)}")

# Tokenizar todo el dataset
# padding=True → todas las frases tendrán la misma longitud (rellena con ceros)
# truncation=True → corta frases demasiado largas
# max_length=128 → máximo 128 tokens (suficiente para titulares)
def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Aplicar a todo el dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print(f"\nDataset tokenizado:")
print(f"  Columnas: {tokenized_dataset['train'].column_names}")
print(f"  Ejemplo de input_ids (primeros 20): {tokenized_dataset['train'][0]['input_ids'][:20]}")
print(f"  Longitud de input_ids: {len(tokenized_dataset['train'][0]['input_ids'])}")

CÓMO TOKENIZA FINBERT:

  Texto:   Profit increased significantly in the third quarter
  Tokens:  ['profit', 'increased', 'significantly', 'in', 'the', 'third', 'quarter']
  IDs:     [101, 5618, 3445, 6022, 1999, 1996, 2353, 4284, 102]
  N tokens: 7

  Texto:   Operating losses widened due to restructuring charges
  Tokens:  ['operating', 'losses', 'widened', 'due', 'to', 'restructuring', 'charges']
  IDs:     [101, 4082, 6409, 8723, 2349, 2000, 18322, 5571, 102]
  N tokens: 7

  Texto:   The annual general meeting will be held next month
  Tokens:  ['the', 'annual', 'general', 'meeting', 'will', 'be', 'held', 'next', 'month']
  IDs:     [101, 1996, 3296, 2236, 3116, 2097, 2022, 2218, 2279, 3204, 102]
  N tokens: 9


Map:   0%|          | 0/3392 [00:00<?, ? examples/s]

Map:   0%|          | 0/727 [00:00<?, ? examples/s]

Map:   0%|          | 0/727 [00:00<?, ? examples/s]


Dataset tokenizado:
  Columnas: ['sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
  Ejemplo de input_ids (primeros 20): [101, 1999, 1996, 11275, 3032, 1010, 4341, 3062, 2011, 4413, 1012, 1020, 1003, 1012, 102, 0, 0, 0, 0, 0]
  Longitud de input_ids: 128


Fine-tuning de FinBERT

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 1. Cargar modelo pre-entrenado
#    num_labels=3 → añade una capa de clasificación con 3 salidas
#    torch_dtype=torch.float32 → evita la detección automática de dtype via mmap
import torch
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.float32,
)

# Contar parámetros (para que veas la escala)
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales:     {total_params:,}")
print(f"Parámetros entrenables: {trainable:,}")
print(f"→ Logistic Regression tenía ~15,000. Este tiene ~110 MILLONES.")

# 2. Métricas de evaluación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

# 3. Configurar entrenamiento
#    Estos son HIPERPARÁMETROS — valores que controlan cómo aprende
training_args = TrainingArguments(
    output_dir="./finbert-sentiment",   # donde guardar checkpoints
    num_train_epochs=3,                 # cuántas veces ver todo el dataset
    per_device_train_batch_size=16,     # frases por paso de entrenamiento
    per_device_eval_batch_size=32,      # frases por paso de evaluación
    learning_rate=2e-5,                 # qué tan rápido aprende (CLAVE)
    weight_decay=0.01,                  # regularización (evita overfitting)
    eval_strategy="epoch",             # evaluar al final de cada época
    save_strategy="epoch",              # guardar modelo al final de cada época
    load_best_model_at_end=True,        # al terminar, cargar el mejor
    metric_for_best_model="f1",         # "mejor" = mayor F1
    logging_steps=50,                   # imprimir métricas cada 50 pasos
    report_to="none",                   # no enviar a WandB/TensorBoard
)

# 4. Crear Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

# 5. ¡ENTRENAR!
print("\n Empezando fine-tuning...")
print("   (Con GPU: ~5 min. Sin GPU: ~30-60 min)")
print("   Verás las métricas actualizarse cada época.\n")

trainer.train()

Evaluar el Transformer

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

# Predecir en test
predictions = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Métricas
print("=" * 60)
print("🏆 RESULTADOS FINBERT")
print("=" * 60)
print(f"\nAccuracy: {accuracy_score(labels, preds):.1%}")
print(f"F1 (weighted): {f1_score(labels, preds, average='weighted'):.1%}")

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=["negative", "neutral", "positive"]))

print("Matriz de confusión:")
cm = confusion_matrix(labels, preds)
label_names = ["negative", "neutral", "positive"]
print(f"              {'neg':>8s}  {'neu':>8s}  {'pos':>8s}")
for i, name in enumerate(label_names):
    row = "  ".join(f"{v:8d}" for v in cm[i])
    print(f"  {name:10s}  {row}")

# Comparación con baseline
print(f"\n{'='*60}")
print("📊 COMPARACIÓN BASELINE vs FINBERT")
print(f"{'='*60}")
print(f"  Logistic Regression:  ~75% accuracy  (tu resultado del bloque 5)")
print(f"  FinBERT:              {accuracy_score(labels, preds):.1%} accuracy")
print(f"\n  → El Transformer entiende CONTEXTO:")
print(f"    'profit fell' = NEGATIVO")
print(f"    'profit rose' = POSITIVO")
print(f"    Logistic Regression los confundía porque ambos tienen 'profit'.")

NameError: name 'trainer' is not defined

Guardar y usar el modelo final

In [ ]:
from transformers import pipeline

# 1. Guardar modelo
trainer.save_model("./finbert-sentiment-final")
tokenizer.save_pretrained("./finbert-sentiment-final")
print(" Modelo guardado en ./finbert-sentiment-final/")

# 2. Cargar como pipeline (la forma más fácil de usar un Transformer)
#    pipeline() abstrae toda la complejidad: tokeniza, predice, decodifica
classifier = pipeline(
    "sentiment-analysis",
    model="./finbert-sentiment-final",
    tokenizer="./finbert-sentiment-final"
)

# 3. Probar con los mismos titulares que fallaban antes
headlines = [
    "Apple reports record quarterly revenue driven by strong iPhone sales",
    "Tesla stock surges 15% after beating earnings expectations",
    "Bank announces massive layoffs affecting 10000 employees",
    "Company shares plunge 20% after fraud investigation announced",
    "The company maintained its current dividend policy",
    "Revenue was flat compared to the previous quarter",
    "Oil prices remain volatile amid geopolitical tensions",
]

emojis = {"negative": "📉", "neutral": "➖", "positive": "📈"}

print("\n🔮 PREDICCIONES FINBERT:")
print("=" * 60)
for h in headlines:
    result = classifier(h)[0]
    emoji = emojis.get(result["label"], "?")
    print(f"  {emoji} {result['label']:8s} ({result['score']:.0%}) │ {h[:60]}")
print()
print("Compara estos resultados con los del Bloque 7.")
print("Deberías ver que los casos que antes fallaban ahora aciertan.")